# Concept Bottleneck Models

## What This Is
Concept Bottleneck Models (CBMs, Koh et al., 2020) add an interpretable intermediate representation: instead of input → label, the model learns input → concepts → label.

**Architecture**:
1. **Concept encoder**: maps input to a vector of human-defined concept scores (e.g., "has_stripes", "has_paws", "is_large")
2. **Task predictor**: maps concept scores to final label

**Benefits**:
- **Interpretability**: Inspect which concepts drove the prediction
- **Intervention**: Correct a concept at test time (human in the loop) and see how it changes the prediction
- **Debugging**: Identify when the model uses wrong concepts

**Limitations**: Requires concept annotations (expensive), concept bottleneck may not capture all necessary information (incomplete concepts), interventions require access to the model's internals.

In [ ]:
import numpy as np
from typing import List, Dict, Tuple

np.random.seed(42)

# Concept Bottleneck Model for animal classification
# Concepts: [has_wings, warm_blooded, has_fur, can_fly, lays_eggs, aquatic]
# Task: classify into bird/mammal/reptile

CONCEPT_NAMES = ['has_wings', 'warm_blooded', 'has_fur', 'can_fly', 'lays_eggs', 'aquatic']
CLASS_NAMES = ['bird', 'mammal', 'reptile']

def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
def softmax(z):
    ez = np.exp(z - z.max(axis=-1, keepdims=True))
    return ez / ez.sum(axis=-1, keepdims=True)

# Generate dataset with concept annotations
def generate_animal_dataset(n_each=100):
    # Visual features: 20-dim noise + signal
    bird_raw = np.random.randn(n_each, 20)
    bird_raw[:, :6] += np.array([2.0, 1.5, -1.5, 1.8, 2.0, 0.5])  # shift means
    
    mammal_raw = np.random.randn(n_each, 20)
    mammal_raw[:, :6] += np.array([-1.5, 2.0, 2.0, -1.5, -1.8, -0.5])
    
    reptile_raw = np.random.randn(n_each, 20)
    reptile_raw[:, :6] += np.array([-1.0, -1.5, -1.5, -1.0, 1.5, 1.0])
    
    # Concept annotations (noisy binary)
    def bird_concepts(n):
        return np.column_stack([
            np.random.binomial(1, 0.9, n),  # has_wings
            np.random.binomial(1, 0.85, n), # warm_blooded
            np.random.binomial(1, 0.05, n), # has_fur
            np.random.binomial(1, 0.75, n), # can_fly
            np.random.binomial(1, 0.9, n),  # lays_eggs
            np.random.binomial(1, 0.15, n), # aquatic
        ])
    def mammal_concepts(n):
        return np.column_stack([
            np.random.binomial(1, 0.05, n),
            np.random.binomial(1, 0.97, n),
            np.random.binomial(1, 0.9, n),
            np.random.binomial(1, 0.08, n),
            np.random.binomial(1, 0.05, n),
            np.random.binomial(1, 0.2, n),
        ])
    def reptile_concepts(n):
        return np.column_stack([
            np.random.binomial(1, 0.05, n),
            np.random.binomial(1, 0.1, n),
            np.random.binomial(1, 0.05, n),
            np.random.binomial(1, 0.1, n),
            np.random.binomial(1, 0.85, n),
            np.random.binomial(1, 0.4, n),
        ])
    
    X = np.vstack([bird_raw, mammal_raw, reptile_raw])
    C = np.vstack([bird_concepts(n_each), mammal_concepts(n_each), reptile_concepts(n_each)])
    y = np.array([0]*n_each + [1]*n_each + [2]*n_each)
    
    return X, C, y

X, C, y = generate_animal_dataset(100)
print(f'Dataset: {len(X)} animals, {X.shape[1]} visual features, {C.shape[1]} concepts')
print(f'Classes: {CLASS_NAMES}')
print(f'Concepts: {CONCEPT_NAMES}')


In [ ]:
# Train Concept Bottleneck Model

# Stage 1: Train concept encoder (visual features -> concept scores)
# Stage 2: Train task predictor (concept scores -> class)

# Split data
perm = np.random.permutation(len(X))
X, C, y = X[perm], C[perm], y[perm]
split = 240
X_tr, C_tr, y_tr = X[:split], C[:split], y[:split]
X_te, C_te, y_te = X[split:], C[split:], y[split:]

# Normalize X
mu_x, s_x = X_tr.mean(0), X_tr.std(0) + 1
X_tr_n = (X_tr - mu_x) / s_x
X_te_n = (X_te - mu_x) / s_x

# Stage 1: Concept encoder (logistic regression per concept)
W_concept = np.random.randn(len(CONCEPT_NAMES), X_tr_n.shape[1]) * 0.1
b_concept = np.zeros(len(CONCEPT_NAMES))
lr = 0.05
for _ in range(300):
    preds = sigmoid(X_tr_n @ W_concept.T + b_concept)  # N x 6
    err = preds - C_tr  # N x 6
    W_concept -= lr * (err.T @ X_tr_n) / len(X_tr_n)
    b_concept -= lr * err.mean(0)

# Get concept predictions
C_pred_tr = sigmoid(X_tr_n @ W_concept.T + b_concept)
C_pred_te = sigmoid(X_te_n @ W_concept.T + b_concept)

# Stage 2: Task predictor (concepts -> class)
W_task = np.random.randn(3, len(CONCEPT_NAMES)) * 0.3
b_task = np.zeros(3)
for _ in range(300):
    logits = C_pred_tr @ W_task.T + b_task
    probs = softmax(logits)
    probs[np.arange(len(y_tr)), y_tr] -= 1
    W_task -= lr * (probs.T @ C_pred_tr) / len(y_tr)
    b_task -= lr * probs.mean(0)

# Evaluate
logits_te = C_pred_te @ W_task.T + b_task
pred_te = np.argmax(logits_te, axis=1)
acc = (pred_te == y_te).mean()
print(f'CBM Test Accuracy: {acc:.3f}')

# Show concept weights for interpretability
print('\nTask predictor weights (concept -> class):')
print(f'{'Concept':<15} {'Bird':>8} {'Mammal':>8} {'Reptile':>8}')
print('-' * 45)
for i, name in enumerate(CONCEPT_NAMES):
    print(f'{name:<15} {W_task[0,i]:>8.3f} {W_task[1,i]:>8.3f} {W_task[2,i]:>8.3f}')

# Demonstrate intervention
sample_idx = np.random.choice(len(X_te))
x_sample = X_te_n[sample_idx]
c_pred = sigmoid(x_sample @ W_concept.T + b_concept)
pred_before = np.argmax(c_pred @ W_task.T + b_task)

# Intervene: set has_wings=1 (correct the model's concept prediction)
c_intervened = c_pred.copy()
c_intervened[0] = 1.0  # force has_wings=1
pred_after = np.argmax(c_intervened @ W_task.T + b_task)

print(f'\nIntervention demo:')
print(f'  True class: {CLASS_NAMES[y_te[sample_idx]]}')
print(f'  Predicted concept has_wings: {c_pred[0]:.3f}')
print(f'  Prediction before intervention: {CLASS_NAMES[pred_before]}')
print(f'  Prediction after has_wings=1:   {CLASS_NAMES[pred_after]}')
